In [6]:
import os
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully.")

Libraries imported successfully.


In [7]:
def calculate_skew_kurt_manual(x):
    n = len(x)
    if n == 0:
        return 0.0, 0.0
    mu = np.mean(x)
    m2 = np.mean((x - mu)**2)
    m3 = np.mean((x - mu)**3)
    m4 = np.mean((x - mu)**4)
    
    if m2 == 0:
        return 0.0, 0.0
    
    skew = m3 / (m2**(3/2))
    kurt = m4 / (m2**2)
    return skew, kurt

def calculate_entropy(x, num_bins=5):
    hist, _ = np.histogram(x, bins=num_bins)
    hist = hist.astype(float)
    hist /= hist.sum()
    hist = hist[hist > 0]
    return -np.sum(hist * np.log2(hist))

def calculate_dominant_freq_fft(x, num_bins=5):
    fft_vals = np.abs(np.fft.rfft(x))
    limited_fft = fft_vals[:num_bins]
    if len(limited_fft) == 0:
        return 0.0
    return np.argmax(limited_fft)

def calculate_spectral_entropy(x, num_bins=5):
    fft_vals = np.abs(np.fft.rfft(x))
    power_spectrum = fft_vals[:num_bins]**2
    if power_spectrum.sum() == 0:
        return 0.0
    prob_spectrum = power_spectrum / power_spectrum.sum()
    prob_spectrum = prob_spectrum[prob_spectrum > 0]
    return -np.sum(prob_spectrum * np.log2(prob_spectrum))

def extract_features_from_channel(x):
    features = []
    n = len(x)
    
    mean_x = np.mean(x)
    std_x = np.std(x, ddof=1)
    min_x = np.min(x)
    max_x = np.max(x)
    range_x = max_x - min_x
    var_x = np.var(x, ddof=1)
    features.extend([mean_x, std_x, min_x, max_x, range_x, var_x])
    
    skew_x, kurt_x = calculate_skew_kurt_manual(x)
    energy_x = np.sum(x**2)
    entropy_x = calculate_entropy(x, num_bins=5)
    features.extend([skew_x, kurt_x, energy_x, entropy_x])
    
    rms_x = np.sqrt(np.mean(x**2))
    zcr_x = np.sum(np.diff(np.sign(x)) != 0) / (len(x) - 1) if len(x) > 1 else 0
    mad_x = np.mean(np.abs(x - mean_x))
    median_x = np.median(x)
    iqr_x = np.percentile(x, 75) - np.percentile(x, 25)
    autocorr_x = np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 1 else 0
    features.extend([rms_x, zcr_x, mad_x, median_x, iqr_x, autocorr_x])
    
    wl_x = np.sum(np.abs(np.diff(x)))
    dom_freq_x = calculate_dominant_freq_fft(x, num_bins=5)
    spec_ent_x = calculate_spectral_entropy(x, num_bins=5)
    std_jerk_x = np.std(np.diff(x)) if len(x) > 1 else 0
    features.extend([wl_x, dom_freq_x, spec_ent_x, std_jerk_x])
    
    return features

def extract_features_window(window_data):
    all_features = []
    for col_idx in range(window_data.shape[1]):
        channel_data = window_data[:, col_idx]
        channel_features = extract_features_from_channel(channel_data)
        all_features.extend(channel_features)
    
    last_sample_raw = window_data[-1, :]
    all_features.extend(last_sample_raw)
    
    return np.array(all_features)

In [8]:
DATA_DIR = './imu_data/'
data_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
print(f"Found {len(data_files)} data files in '{DATA_DIR}'.")

label_map = {
    'sitting': 0,
    'standing': 1,
    'walking': 2,
    'brisk_walking': 3,
    'jogging': 4,
    'cycling': 5,
    'stair_up': 6,
    'stair_down': 7,
    'sit_stand_sit': 8,
    'phone_interaction': 9,
    'eating_with_spoon': 10,
    'pick_and_place': 11
}

reverse_label_map = {v: k for k, v in label_map.items()}

all_features = []
all_labels = []

for file in data_files:
    filepath = os.path.join(DATA_DIR, file)
    try:
        df = pd.read_csv(filepath)
        if 'activity' in df.columns:
            activity_col_val = df['activity'].iloc[0]
            activity_name = activity_col_val.strip().lower()
        else:
            base_name = os.path.splitext(file)[0]
            base_name = re.sub(r'_seg\d+', '', base_name)
            activity_name = None
            parts = base_name.split('_')
            for i in range(len(parts), 0, -1):
                candidate = '_'.join(parts[:i])
                if candidate in label_map:
                    activity_name = candidate
                    break
            if activity_name is None:
                print(f"Warning: Unknown activity for file '{file}' based on filename -> skipping")
                continue
        
        label = label_map.get(activity_name, -1)
        if label == -1:
            print(f"Warning: Activity '{activity_name}' not in label_map (file: {file}) -> skipping")
            continue

        window_size = 20
        overlap = 0.5
        step = int(window_size * (1 - overlap))

        imu_cols = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
        imu_cols = [col for col in imu_cols if col in df.columns]
        if len(imu_cols) != 6:
            print(f"Error: Missing IMU columns in {file}. Found: {imu_cols} -> skipping")
            continue
        raw_imu_data = df[imu_cols].values

        for start in range(0, len(raw_imu_data) - window_size + 1, step):
            win = raw_imu_data[start:start+window_size]
            window_features = extract_features_window(win)
            all_features.append(window_features)
            all_labels.append(label)
            
    except Exception as e:
        print(f"Error processing file {file}: {e}")

print(f"Processed {len(all_features)} windows after applying sliding window and feature extraction.")

X = np.array(all_features)
y = np.array(all_labels)

if X.size == 0:
    raise ValueError("No valid data was loaded. Check DATA_DIR and file formats.")

print(f"Final feature matrix shape (X): {X.shape}")
print(f"Final label vector shape (y): {y.shape}")
print(f"Unique labels found in  {sorted(np.unique(y))}")
label_counts = dict(zip(*np.unique(y, return_counts=True)))
print(f"Label counts: {label_counts}")
print(f"Labels present: {[reverse_label_map.get(k, f'UNKNOWN_{k}') for k in sorted(label_counts.keys())]}")


Found 15 data files in './imu_data/'.
Processed 19642 windows after applying sliding window and feature extraction.
Final feature matrix shape (X): (19642, 126)
Final label vector shape (y): (19642,)
Unique labels found in  [np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
Label counts: {np.int64(0): np.int64(1795), np.int64(2): np.int64(1789), np.int64(3): np.int64(1789), np.int64(4): np.int64(1785), np.int64(5): np.int64(1791), np.int64(6): np.int64(1720), np.int64(7): np.int64(1793), np.int64(8): np.int64(1793), np.int64(9): np.int64(1797), np.int64(10): np.int64(1795), np.int64(11): np.int64(1795)}
Labels present: ['sitting', 'walking', 'brisk_walking', 'jogging', 'cycling', 'stair_up', 'stair_down', 'sit_stand_sit', 'phone_interaction', 'eating_with_spoon', 'pick_and_place']


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape (X_train): {X_train.shape}")
print(f"Test set shape (X_test): {X_test.shape}")

train_label_counts = dict(zip(*np.unique(y_train, return_counts=True)))
test_label_counts = dict(zip(*np.unique(y_test, return_counts=True)))
print(f"Training label distribution: {train_label_counts}")
print(f"Test label distribution: {test_label_counts}")

Training set shape (X_train): (15713, 126)
Test set shape (X_test): (3929, 126)
Training label distribution: {np.int64(0): np.int64(1436), np.int64(2): np.int64(1431), np.int64(3): np.int64(1431), np.int64(4): np.int64(1428), np.int64(5): np.int64(1433), np.int64(6): np.int64(1376), np.int64(7): np.int64(1434), np.int64(8): np.int64(1434), np.int64(9): np.int64(1438), np.int64(10): np.int64(1436), np.int64(11): np.int64(1436)}
Test label distribution: {np.int64(0): np.int64(359), np.int64(2): np.int64(358), np.int64(3): np.int64(358), np.int64(4): np.int64(357), np.int64(5): np.int64(358), np.int64(6): np.int64(344), np.int64(7): np.int64(359), np.int64(8): np.int64(359), np.int64(9): np.int64(359), np.int64(10): np.int64(359), np.int64(11): np.int64(359)}


In [10]:
models_config = {
    "DecisionTree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "criterion": ["gini", "entropy"],
            "max_depth": [6, 8, 10, 12, None],
            "min_samples_split": [5, 10, 15],
            "min_samples_leaf": [2, 4, 6],
            "max_features": ["sqrt", 0.5, None],
            "ccp_alpha": [0.0, 0.001, 0.01]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100],
            "criterion": ["gini", "entropy"],
            "max_depth": [6, 8, 10],
            "min_samples_split": [5, 10],
            "min_samples_leaf": [2, 4],
            "max_features": ["sqrt", 0.5]
        }
    },
    "SVM": {
        "model": SVC(random_state=42),
        "params": {
            "kernel": ["linear", "rbf"],
            "C": [0.1, 1, 10],
            "gamma": ["scale", "auto"]
        }
    },
    "LogisticRegression": {
        "model": LogisticRegression(random_state=42, max_iter=1000, multi_class='ovr'),
        "params": {
            "penalty": ["l1", "l2"],
            "C": [0.1, 1, 10],
            "solver": ["liblinear"]
        }
    }
}

best_models = {}
results_summary = []

for name, config in models_config.items():
    print(f"\n--- Tuning {name} ---")
    grid_search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=StratifiedKFold(n_splits=5),
        scoring="accuracy",
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    best_models[name] = best_model
    
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Best params for {name}: {grid_search.best_params_}")
    print(f"Test Accuracy ({name}): {acc:.4f}")
    print(f"Test F1 Macro ({name}): {f1_macro:.4f}")
    print(f"Test F1 Weighted ({name}): {f1_weighted:.4f}")
    print(f"Confusion Matrix ({name}):\n{cm}")
    
    results_summary.append({
        "Model": name, 
        "Accuracy": acc, 
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted,
        "Best_Params": grid_search.best_params_
    })


--- Tuning DecisionTree ---
Best params for DecisionTree: {'ccp_alpha': 0.0, 'criterion': 'entropy', 'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 4, 'min_samples_split': 15}
Test Accuracy (DecisionTree): 0.9310
Test F1 Macro (DecisionTree): 0.9306
Test F1 Weighted (DecisionTree): 0.9310
Confusion Matrix (DecisionTree):
[[357   0   0   0   0   0   0   2   0   0   0]
 [  0 316   8   0   0  15  16   3   0   0   0]
 [  0  11 332   2   1   6   1   4   0   1   0]
 [  0   0   0 356   0   0   0   0   0   1   0]
 [  0   0   2   0 353   0   0   3   0   0   0]
 [  0  13  10   1   0 288  17   8   0   6   1]
 [  0  28   1   0   0  25 293  12   0   0   0]
 [  0  10   4   1   7  17  12 306   0   0   2]
 [  0   0   0   0   0   0   0   0 358   1   0]
 [  0   0   0   2   0   0   0   0   0 351   6]
 [  2   0   0   0   0   7   0   0   0   2 348]]

--- Tuning RandomForest ---
Best params for RandomForest: {'criterion': 'entropy', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 

In [ ]:
results_df = pd.DataFrame(results_summary)
print("\n--- Model Comparison Summary ---")
print(results_df.to_string(index=False))

In [ ]:
optimal_model_name = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
optimal_model = best_models[optimal_model_name]
print(f"\nOptimal Model selected based on highest Test Accuracy: {optimal_model_name}")

final_y_pred = optimal_model.predict(X_test)
final_acc = accuracy_score(y_test, final_y_pred)
final_cm = confusion_matrix(y_test, final_y_pred)
final_f1_macro = f1_score(y_test, final_y_pred, average='macro')
final_f1_weighted = f1_score(y_test, final_y_pred, average='weighted')
final_cr = classification_report(
    y_test, 
    final_y_pred, 
    target_names=[reverse_label_map[i] for i in sorted(reverse_label_map.keys())], 
    digits=3,
    zero_division=0
)

print(f"\n--- Final Evaluation of Optimal Model ({optimal_model_name}) ---")
print(f"Accuracy: {final_acc:.4f}")
print(f"F1 Score (Macro): {final_f1_macro:.4f}")
print(f"F1 Score (Weighted): {final_f1_weighted:.4f}")
print(f"Confusion Matrix:\n{final_cm}")
print(f"Detailed Classification Report:\n{final_cr}")

In [ ]:
dt_model = best_models.get("DecisionTree")

if dt_model is not None:
    print("\n--- Generating MicroPython Inference Code for Decision Tree ---")
    tree = dt_model.tree_

    def recurse_print_tree(node_id):
        left_child = tree.children_left[node_id]
        right_child = tree.children_right[node_id]

        if left_child == right_child:
            class_counts = tree.value[node_id][0]
            predicted_class = int(np.argmax(class_counts))
            return f"        return {predicted_class} # Leaf node, pred={predicted_class}"

        feature_idx = tree.feature[node_id]
        threshold = tree.threshold[node_id]

        left_code = recurse_print_tree(left_child)
        right_code = recurse_print_tree(right_child)

        code = f"    if features[{feature_idx}] <= {threshold:.6f}" + ":\n"
        for line in left_code.split('\n'):
            code += f"        {line}\n"
        code += f"    else" + ":\n"
        for line in right_code.split('\n'):
            code += f"        {line}\n"
        return code

    tree_code = recurse_print_tree(0)

    n_classes = len(reverse_label_map)
    activity_map_str = "{\n"
    for i in range(n_classes):
        activity_map_str += f"    {i}: \"{reverse_label_map[i]}\",\n"
    activity_map_str += "}"

    micropython_code = f"""
def predict_activity(features):
{tree_code}
ACTIVITY_MAP = {activity_map_str}
"""
    print(micropython_code)

else:
    print("\n--- Warning: Decision Tree model not found to generate MicroPython code. ---")

In [ ]:
dt_model = best_models.get("DecisionTree")

if dt_model is not None:
    print("\n--- Generating Complete Deployable MicroPython Code ---")
    tree = dt_model.tree_

    def recurse_print_tree(node_id):
        left_child = tree.children_left[node_id]
        right_child = tree.children_right[node_id]

        if left_child == right_child:
            class_counts = tree.value[node_id][0]
            predicted_class = int(np.argmax(class_counts))
            return f"        return {predicted_class} # Leaf node, pred={predicted_class}"

        feature_idx = tree.feature[node_id]
        threshold = tree.threshold[node_id]

        left_code = recurse_print_tree(left_child)
        right_code = recurse_print_tree(right_child)

        code = f"    if features[{feature_idx}] <= {threshold:.6f}:\n"
        for line in left_code.split('\n'):
            code += f"        {line}\n"
        code += f"    else:\n"
        for line in right_code.split('\n'):
            code += f"        {line}\n"
        return code

    tree_logic_code = recurse_print_tree(0)

    n_classes = len(reverse_label_map)
    activity_map_py = "{\n"
    for i in range(n_classes):
        activity_map_py += f"    {i}: \"{reverse_label_map[i]}\",\n"
    activity_map_py += "}"

    complete_micropython_content = f'''
import time
from machine import Pin, SPI, LED
from lsm6dsox import LSM6DSOX
import math

spi = SPI(5)
cs = Pin("PF6", Pin.OUT_PP, Pin.PULL_UP)
lsm = LSM6DSOX(spi, cs=cs)
red_led = LED("LED_RED")
green_led = LED("LED_GREEN")
blue_led = LED("LED_BLUE")

def predict_activity_features(features):
{tree_logic_code}

def calculate_skew_manual(x_list):
    n = len(x_list)
    if n <= 1:
        return 0.0
    mean_x = sum(x_list) / n
    diffs = [xi - mean_x for xi in x_list]
    m2 = sum(d ** 2 for d in diffs) / (n - 1)
    m3 = sum(d ** 3 for d in diffs) / n
    if m2 == 0:
        return 0.0
    return m3 / (m2 ** 1.5)

def calculate_kurt_manual(x_list):
    n = len(x_list)
    if n <= 3:
        return 0.0
    mean_x = sum(x_list) / n
    diffs = [xi - mean_x for xi in x_list]
    m2 = sum(d ** 2 for d in diffs) / (n - 1)
    m4 = sum(d ** 4 for d in diffs) / n
    if m2 == 0:
        return 0.0
    return (m4 / (m2 ** 2)) - 3

def calculate_entropy_manual(x_list, num_bins=5):
    min_x = min(x_list)
    max_x = max(x_list)
    if min_x == max_x:
        return 0.0 # No variance, entropy is 0
    bin_width = (max_x - min_x) / num_bins
    if bin_width == 0:
        return 0.0
    histogram = [0] * num_bins
    for val in x_list:
        bin_idx = int((val - min_x) / bin_width)
        if bin_idx >= num_bins:
            bin_idx = num_bins - 1 # Handle edge case where val == max_x
        histogram[bin_idx] += 1
    total = sum(histogram)
    if total == 0:
        return 0.0
    probabilities = [h / total for h in histogram if h > 0]
    entropy = -sum(p * math.log2(p) for p in probabilities if p > 0)
    return entropy

def calculate_dominant_freq_fft(x_list, num_bins=5):
    n = len(x_list)
    if n == 0 or num_bins <= 0:
        return 0.0

    effective_num_bins = min(num_bins, n) 

    bin_magnitudes = []
    for k in range(effective_num_bins):
        real_part = 0.0
        imag_part = 0.0
        for t in range(n):
            angle = -2.0 * math.pi * k * t / n
            cos_val = math.cos(angle)
            sin_val = math.sin(angle)
            real_part += x_list[t] * cos_val
            imag_part += x_list[t] * sin_val

        magnitude = math.sqrt(real_part**2 + imag_part**2)
        bin_magnitudes.append(magnitude)

    max_magnitude = bin_magnitudes[0]
    dominant_bin_index = 0
    for i in range(1, len(bin_magnitudes)):
        if bin_magnitudes[i] > max_magnitude:
            max_magnitude = bin_magnitudes[i]
            dominant_bin_index = i

    return float(dominant_bin_index)

def calculate_spectral_entropy_manual(x_list, num_bins=5):
    n = len(x_list)
    if n == 0 or num_bins <= 0:
        return 0.0

    effective_num_bins = min(num_bins, n)
    
    bin_magnitudes = []
    for k in range(effective_num_bins):
        real_part = 0.0
        imag_part = 0.0
        for t in range(n):
            angle = -2.0 * math.pi * k * t / n
            cos_val = math.cos(angle)
            sin_val = math.sin(angle)
            real_part += x_list[t] * cos_val
            imag_part += x_list[t] * sin_val

        magnitude = math.sqrt(real_part**2 + imag_part**2)
        bin_magnitudes.append(magnitude)
    
    power_spectrum = [mag**2 for mag in bin_magnitudes]
    
    total_power = sum(power_spectrum)
    
    if total_power == 0.0:
        return 0.0
        
    probabilities = [power / total_power for power in power_spectrum]
    
    spectral_entropy = 0.0
    for p in probabilities:
        if p > 0.0:
            spectral_entropy -= p * math.log2(p)
            
    return spectral_entropy

def calculate_zero_crossing_rate(x_list):
    if len(x_list) <= 1:
        return 0.0
    crossings = 0
    prev_sign = 1 if x_list[0] >= 0 else -1
    for i in range(1, len(x_list)):
        curr_sign = 1 if x_list[i] >= 0 else -1
        if curr_sign != prev_sign:
            crossings += 1
        prev_sign = curr_sign
    return crossings / (len(x_list) - 1)

def calculate_autocorr_lag1(x_list):
    if len(x_list) <= 1:
        return 0.0
    mean_x = sum(x_list) / len(x_list)
    numerator = sum((x_list[i] - mean_x) * (x_list[i+1] - mean_x) for i in range(len(x_list)-1))
    denominator = sum((x_i - mean_x)**2 for x_i in x_list)
    if denominator == 0:
        return 0.0
    return numerator / denominator

def extract_features_from_channel_manual(x_list):
    features = []
    n = len(x_list)

    mean_x = sum(x_list) / n
    std_x = math.sqrt(sum((xi - mean_x)**2 for xi in x_list) / (n - 1)) if n > 1 else 0.0
    min_x = min(x_list)
    max_x = max(x_list)
    range_x = max_x - min_x
    var_x = sum((xi - mean_x)**2 for xi in x_list) / (n - 1) if n > 1 else 0.0

    features.extend([
        mean_x,
        std_x,
        min_x,
        max_x,
        range_x,
        var_x
    ])

    skew_x = calculate_skew_manual(x_list)
    kurt_x = calculate_kurt_manual(x_list)
    energy_x = sum(xi**2 for xi in x_list)
    entropy_x = calculate_entropy_manual(x_list, num_bins=5)

    features.extend([
        skew_x,
        kurt_x,
        energy_x,
        entropy_x
    ])

    rms_x = math.sqrt(sum(xi**2 for xi in x_list) / n)
    zcr_x = calculate_zero_crossing_rate(x_list)
    mad_x = sum(abs(xi - mean_x) for xi in x_list) / n
    sorted_x = sorted(x_list) # Note: Sorting might be expensive for N=20, acceptable?
    median_x = sorted_x[n // 2] if n % 2 == 1 else (sorted_x[n // 2 - 1] + sorted_x[n // 2]) / 2
    q75_idx = int(0.75 * (n - 1))
    q25_idx = int(0.25 * (n - 1))
    iqr_x = sorted_x[q75_idx] - sorted_x[q25_idx]
    autocorr_x = calculate_autocorr_lag1(x_list)

    features.extend([
        rms_x,
        zcr_x,
        mad_x,
        median_x,
        iqr_x,
        autocorr_x
    ])

    jerk_diffs = [(x_list[i] - x_list[i-1]) for i in range(1, n)] if n > 1 else [0.0]
    std_jerk_x = math.sqrt(sum(d**2 for d in jerk_diffs) / (len(jerk_diffs) - 1)) if len(jerk_diffs) > 1 else 0.0

    wl_sum = 0.0
    for i in range(1, n):
        wl_sum += abs(x_list[i] - x_list[i-1])
    waveform_length_x = wl_sum

    features.extend([
        waveform_length_x,
        0.0,
        0.0,
        std_jerk_x
    ])

    return features

def extract_features_window_manual(window_data_list_of_lists):
    all_features = []
    for col_idx in range(6):
        channel_data = [window_data_list_of_lists[row_idx][col_idx] for row_idx in range(20)]
        channel_features = extract_features_from_channel_manual(channel_data)
        all_features.extend(channel_features)

    # Append raw values from the 20th (last) sample (indices 0-5 -> Ax-Az, Gx-Gz)
    last_sample_raw = window_data_list_of_lists[-1] # This is the last list of 6 values
    all_features.extend(last_sample_raw)

    return all_features


ACTIVITY_MAP = {activity_map_py}

WINDOW_SIZE = 20
OVERLAP_FRAC = 0.5
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP_FRAC))

imu_buffer = [] # Stores lists of [ax, ay, az, gx, gy, gz]

print("Nicla Vision: Multi-Class IMU-based Activity Recognition Started (Deployed Model)")

try:
    while True:
        ax, ay, az = lsm.accel()
        gx, gy, gz = lsm.gyro()
        current_sample = [ax, ay, az, gx, gy, gz]

        imu_buffer.append(current_sample)

        if len(imu_buffer) >= WINDOW_SIZE:
            window_data = imu_buffer[-WINDOW_SIZE:]

            features_132 = extract_features_window_manual(window_data)

            predicted_label_int = predict_activity_features(features_132)

            predicted_activity_str = ACTIVITY_MAP.get(predicted_label_int, "unknown")

            red_led.off()
            green_led.off()
            blue_led.off()
            
            if predicted_label_int in [0, 1]: # sitting, standing
                green_led.on()
            elif predicted_label_int in [2, 3, 4]: # walking, brisk_walking, jogging
                red_led.on()
            else:
                blue_led.on()

            print(
                "Accelerometer: x:{{:>8.3f}} y:{{:>8.3f}} z:{{:>8.3f}} | "
                "Gyroscope: x:{{:>8.3f}} y:{{:>8.3f}} z:{{:>8.3f}} | "
                "Pred Label: {{}} | Pred Name: {{}}".format(
                    ax, ay, az,
                    gx, gy, gz,
                    predicted_label_int,
                    predicted_activity_str
                )
            )

            if len(imu_buffer) > WINDOW_SIZE - STEP_SIZE:
                 imu_buffer = imu_buffer[-(WINDOW_SIZE - STEP_SIZE):]

except KeyboardInterrupt:
    red_led.off()
    green_led.off()
    blue_led.off()
    print("Multi-Class Activity Recognition Stopped")

'''

    output_filename = "multi_class_predict.py"
    with open(output_filename, "w") as f:
        f.write(complete_micropython_content)
    print(f"\n--- Generated Complete Deployable MicroPython code saved to '{output_filename}' ---")
    print("--- This file includes the trained model, feature extraction, and hardware loop. ---")

else:
    print("\n--- Skipped saving MicroPython code as Decision Tree model was not found. ---")
    print("--- Ensure the training notebook was run successfully before executing this cell. ---")
